# 05 — Africa long-run supplement (1997–2025)

**Question.** Does the 2014–2025 cross-region story hold up when we look at Africa back to the late-1990s, where ACLED coverage is deepest? What structural breaks does the 2014 cut hide?

**Method.** Load the pre-filter Africa frame; restrict to complete calendar years (1997–2025); repeat RQ1 (events vs fatalities trends, correlations, log-slopes) and RQ2 (shift-share decomposition of aggregate lethality) on this longer series; add a country-level choropleth of cumulative African activity.

**Takeaway.** Reported at the bottom. This supplement is Africa-only and kept separate from the cross-region comparison.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

PROJECT = Path('/Users/zhangtingmin/Desktop/PhD/Year_1_Sem_2/POLI_3148/Assignment 1 copy 5')
DERIVED = PROJECT / 'data' / 'derived'
FIGDIR = PROJECT / 'docs' / 'figures'

af = pd.read_parquet(DERIVED / 'acled_africa_full.parquet')
af = af[(af['WEEK'] >= '1997-01-01') & (af['WEEK'] <= '2025-12-31')].copy()
af['YEAR'] = af['WEEK'].dt.year
print('Africa rows:', len(af), 'years:', af['YEAR'].min(), '-', af['YEAR'].max())

Africa rows: 262744 years: 1997 - 2025


## Africa map: cumulative events and fatalities by country, 1997–2025

In [2]:
by_country = af.groupby('COUNTRY')[['EVENTS','FATALITIES']].sum().reset_index()
fig_map = make_subplots(rows=1, cols=2,
                        specs=[[{'type':'choropleth'}, {'type':'choropleth'}]],
                        subplot_titles=('Cumulative EVENTS 1997–2025','Cumulative FATALITIES 1997–2025'))
for i, (col, cs, title) in enumerate([
    ('EVENTS','Blues','log10(events)'), ('FATALITIES','Reds','log10(fatalities)')]):
    fig_map.add_trace(go.Choropleth(
        locations=by_country['COUNTRY'], locationmode='country names',
        z=np.log10(by_country[col].clip(lower=1)),
        colorscale=cs,
        colorbar=dict(title=title, x=0.46 if i==0 else 1.02),
        customdata=np.stack([by_country['COUNTRY'], by_country[col]], axis=-1),
        hovertemplate='%{customdata[0]}<br>'+col+': %{customdata[1]:,}<extra></extra>'
    ), row=1, col=i+1)
    fig_map.update_geos(scope='africa', row=1, col=i+1)
fig_map.update_layout(title='Africa — cumulative ACLED activity by country, 1997–2025',
                      height=520, margin=dict(l=10,r=10,t=80,b=10))
fig_map.write_html(FIGDIR / '05_africa_map.html', include_plotlyjs='cdn')
fig_map.show()

## RQ1 on long-run Africa

In [3]:
annual = af.groupby('YEAR')[['EVENTS','FATALITIES']].sum().reset_index()
annual['LETHALITY'] = annual['FATALITIES']/annual['EVENTS']
annual['EVENTS_YOY'] = annual['EVENTS'].pct_change()
annual['FATALITIES_YOY'] = annual['FATALITIES'].pct_change()
annual

,YEAR,EVENTS,FATALITIES,LETHALITY,EVENTS_YOY,FATALITIES_YOY
0,1997,3135,27042,8.625837,NaN,NaN
1,1998,4444,72438,16.300180,0.417544,1.678722
2,1999,4714,159821,33.903479,0.060756,1.206314
3,2000,4159,23919,5.751142,-0.117734,-0.850339
4,2001,3585,27492,7.668619,-0.138014,0.149379
5,2002,4173,28039,6.719147,0.164017,0.019897
6,2003,3636,21519,5.918317,-0.128684,-0.232533
7,2004,3082,19660,6.378975,-0.152365,-0.086389
8,2005,2913,8201,2.815311,-0.054835,-0.582859
9,2006,2705,8019,2.964510,-0.071404,-0.022192


In [4]:
fig_dual = make_subplots(specs=[[{'secondary_y':True}]])
fig_dual.add_trace(go.Scatter(x=annual['YEAR'], y=annual['EVENTS'], name='Events',
                              line=dict(color='steelblue', width=3), mode='lines+markers'), secondary_y=False)
fig_dual.add_trace(go.Scatter(x=annual['YEAR'], y=annual['FATALITIES'], name='Fatalities',
                              line=dict(color='firebrick', width=3), mode='lines+markers'), secondary_y=True)
fig_dual.update_yaxes(title_text='Events', secondary_y=False)
fig_dual.update_yaxes(title_text='Fatalities', secondary_y=True)
fig_dual.update_layout(title='Africa annual events vs fatalities, 1997–2025',
                       height=450, hovermode='x unified', xaxis=dict(dtick=2))
fig_dual.write_html(FIGDIR / '05_africa_dual_axis.html', include_plotlyjs='cdn')
fig_dual.show()

In [5]:
fig_l = go.Figure()
fig_l.add_trace(go.Scatter(x=annual['YEAR'], y=annual['LETHALITY'],
                           mode='lines+markers', line=dict(color='purple', width=3)))
fig_l.add_vline(x=2011, line_dash='dash', line_color='grey',
                annotation_text='Arab Spring / Libya', annotation_position='top')
fig_l.add_vline(x=2014, line_dash='dash', line_color='grey',
                annotation_text='2014 cutoff', annotation_position='bottom')
fig_l.update_layout(title='Africa lethality (fatalities/event), 1997–2025',
                    xaxis_title='Year', yaxis_title='Fatalities per event',
                    height=450, xaxis=dict(dtick=2))
fig_l.write_html(FIGDIR / '05_africa_lethality.html', include_plotlyjs='cdn')
fig_l.show()

In [6]:
x = annual['YEAR'].to_numpy()
ev = stats.linregress(x, np.log(annual['EVENTS']))
ft = stats.linregress(x, np.log(annual['FATALITIES']))
pear = stats.pearsonr(annual['EVENTS'], annual['FATALITIES'])
spear = stats.spearmanr(annual['EVENTS'], annual['FATALITIES'])
print(f'Pearson r={pear.statistic:.3f} p={pear.pvalue:.3g}')
print(f'Spearman r={spear.statistic:.3f} p={spear.pvalue:.3g}')
print(f'log(EV)  slope={ev.slope:.4f}  -> {(np.exp(ev.slope)-1)*100:.1f}%/yr  (SE={ev.stderr:.4f})')
print(f'log(FT)  slope={ft.slope:.4f}  -> {(np.exp(ft.slope)-1)*100:.1f}%/yr  (SE={ft.stderr:.4f})')
# Split-window: pre/post 2014
for y0,y1 in [(1997,2013),(2014,2025)]:
    sub = annual[(annual['YEAR']>=y0)&(annual['YEAR']<=y1)]
    e = stats.linregress(sub['YEAR'], np.log(sub['EVENTS']))
    f = stats.linregress(sub['YEAR'], np.log(sub['FATALITIES']))
    print(f'  {y0}-{y1}: ev={(np.exp(e.slope)-1)*100:.1f}%/yr  ft={(np.exp(f.slope)-1)*100:.1f}%/yr')

Pearson r=0.412 p=0.0264
Spearman r=0.738 p=4.91e-06
log(EV)  slope=0.1135  -> 12.0%/yr  (SE=0.0085)
log(FT)  slope=0.0272  -> 2.8%/yr  (SE=0.0154)
  1997-2013: ev=5.5%/yr  ft=-7.5%/yr
  2014-2025: ev=13.6%/yr  ft=8.4%/yr


## RQ2 on long-run Africa: shift-share decomposition

In [7]:
def shift_share(df_long, year0, year1, group_col):
    d0 = df_long[df_long['YEAR']==year0].set_index(group_col)
    d1 = df_long[df_long['YEAR']==year1].set_index(group_col)
    keys = sorted(set(d0.index) | set(d1.index))
    d0 = d0.reindex(keys).fillna(0); d1 = d1.reindex(keys).fillna(0)
    s0 = d0['EVENTS']/max(d0['EVENTS'].sum(),1); s1 = d1['EVENTS']/max(d1['EVENTS'].sum(),1)
    l0 = np.where(d0['EVENTS']>0, d0['FATALITIES']/d0['EVENTS'].replace(0,np.nan), 0)
    l1 = np.where(d1['EVENTS']>0, d1['FATALITIES']/d1['EVENTS'].replace(0,np.nan), 0)
    L0 = float((s0*l0).sum()); L1 = float((s1*l1).sum())
    return L0, L1, float(((s1-s0)*l0).sum()), float((s0*(l1-l0)).sum()), float(((s1-s0)*(l1-l0)).sum())

by_type = af.groupby(['YEAR','EVENT_TYPE'])[['EVENTS','FATALITIES']].sum().reset_index()

print('Full window 1997 → 2025:')
L0,L1,c,w,i = shift_share(by_type, 1997, 2025, 'EVENT_TYPE')
dL = L1-L0
print(f'  L0={L0:.3f}  L1={L1:.3f}  dL={dL:+.3f}')
print(f'  comp={c:+.3f} ({c/dL*100:+.1f}%)  within={w:+.3f} ({w/dL*100:+.1f}%)  inter={i:+.3f} ({i/dL*100:+.1f}%)')

for (y0,y1) in [(1997,2013),(2014,2025)]:
    L0,L1,c,w,i = shift_share(by_type, y0, y1, 'EVENT_TYPE')
    dL = L1-L0
    print(f'\nSub-window {y0} → {y1}:')
    print(f'  L0={L0:.3f}  L1={L1:.3f}  dL={dL:+.3f}')
    if dL != 0:
        print(f'  comp={c:+.3f} ({c/dL*100:+.1f}%)  within={w:+.3f} ({w/dL*100:+.1f}%)  inter={i:+.3f} ({i/dL*100:+.1f}%)')

Full window 1997 → 2025:
  L0=8.626  L1=1.418  dL=-7.208
  comp=-1.877 (+26.0%)  within=-6.844 (+94.9%)  inter=+1.512 (-21.0%)

Sub-window 1997 → 2013:
  L0=8.626  L1=2.133  dL=-6.493
  comp=-1.899 (+29.2%)  within=-6.000 (+92.4%)  inter=+1.405 (-21.6%)

Sub-window 2014 → 2025:
  L0=2.438  L1=1.418  dL=-1.020
  comp=-0.250 (+24.6%)  within=-0.824 (+80.8%)  inter=+0.054 (-5.3%)


In [8]:
years = sorted(by_type['YEAR'].unique())
rows = []
for y0,y1 in zip(years[:-1], years[1:]):
    L0,L1,c,w,i = shift_share(by_type, y0, y1, 'EVENT_TYPE')
    rows.append({'YEAR':y1, 'dL':L1-L0, 'comp':c, 'within':w, 'inter':i})
roll = pd.DataFrame(rows)

fig_roll = go.Figure()
fig_roll.add_trace(go.Bar(x=roll['YEAR'], y=roll['comp'],   name='Composition', marker_color='steelblue'))
fig_roll.add_trace(go.Bar(x=roll['YEAR'], y=roll['within'], name='Within-type', marker_color='firebrick'))
fig_roll.add_trace(go.Bar(x=roll['YEAR'], y=roll['inter'],  name='Interaction', marker_color='grey'))
fig_roll.add_trace(go.Scatter(x=roll['YEAR'], y=roll['dL'], name='ΔL (total)',
                              mode='lines+markers', line=dict(color='black')))
fig_roll.update_layout(title='Africa year-on-year shift-share decomposition, 1998–2025',
                       barmode='relative', xaxis_title='Year', yaxis_title='ΔL',
                       height=480, xaxis=dict(dtick=2))
fig_roll.write_html(FIGDIR / '05_africa_rolling_decomp.html', include_plotlyjs='cdn')
fig_roll.show()

## Takeaway

Interpretation deferred to the report.